# 07 — Leakage-Resistant Model Development and Robustness Evaluation

**Dataset:** Kaggle *Metaverse Financial Transactions Dataset*  
**Kaggle source:** https://www.kaggle.com/datasets/faizaniftikharjanjua/metaverse-financial-transactions-dataset  
**Expected source shape:** 78,600 rows × 14 columns  
**Target:** `anomaly` (`low_risk`, `moderate_risk`, `high_risk`)

## Purpose

Develop a more defensible baseline using features that are less obviously tied to the target-generation pattern, then evaluate it with chronological train/validation/test partitions, class-aware metrics, high-risk threshold analysis, and robustness checks.

This notebook builds on the previous audit:
- `risk_score` showed perfect target alignment and is excluded.
- `transaction_type` produced perfect classification in the prior experiment and is excluded from the primary model.
- `age_group` and `purchase_pattern` were perfectly paired in the observed data and are excluded from the primary model.
- Sender and receiver addresses are repeated identifiers and are excluded from ordinary predictive inputs.
- `timestamp` is used for ordering and split construction, not as a raw model feature.

## Important research limitations

The target is the Kaggle-provided risk category. These experiments predict that dataset label; they do **not** establish detection of independently verified real-world fraud. Excluding suspicious fields reduces obvious leakage risk but cannot prove that the remaining fields were independently generated. Document provenance and prediction-time availability before making strong claims.

**Do not change feature sets or model settings after seeing test results.** The test partition is held out for final evaluation.

## 1. Load the original dataset and verify its identity

In [ ]:
from pathlib import Path
import hashlib, json, warnings, sys, subprocess, importlib.util
import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = Path("metaverse_transactions_dataset.csv")
if not DATA_PATH.exists():
    candidates = [Path("metaverse_transactions_dataset(1).csv")] + list(Path(".").glob("*.csv"))
    DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Upload the original Kaggle CSV to this Colab session or edit DATA_PATH."
    )

source_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
df = pd.read_csv(DATA_PATH)

TARGET = "anomaly"
if TARGET not in df.columns:
    raise KeyError(f"Target column {TARGET!r} not found. Columns: {df.columns.tolist()}")

print("File:", DATA_PATH.name)
print("SHA-256:", source_sha256)
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
display(df[TARGET].value_counts(dropna=False).rename_axis("class").to_frame("count"))

## 2. Define the conservative primary feature set

In [ ]:
# Primary features exclude fields that were strongly suspicious in earlier audits.
EXCLUDED_FIELDS = [
    TARGET, "risk_score", "transaction_type", "age_group", "purchase_pattern",
    "sending_address", "receiving_address", "timestamp"
]

PRIMARY_FEATURES_REQUESTED = [
    "amount",
    "hour_of_day",
    "login_frequency",
    "session_duration",
    "location_region",
    "ip_prefix",
]

missing_features = [c for c in PRIMARY_FEATURES_REQUESTED if c not in df.columns]
if missing_features:
    raise ValueError(f"Expected primary features are missing: {missing_features}")

FEATURES = PRIMARY_FEATURES_REQUESTED.copy()
print("Primary features:", FEATURES)
print("Excluded fields:", EXCLUDED_FIELDS)
display(df[FEATURES].head())

## 3. Create chronological train / validation / test partitions

In [ ]:
# Chronological split: earliest 70% train, next 10% validation, latest 20% test.
# Validation is used for model comparison / threshold selection.
# Test is not used for tuning.
work = df.copy()
work["_parsed_timestamp"] = pd.to_datetime(work["timestamp"], errors="coerce", utc=True)
invalid_timestamp_rows = int(work["_parsed_timestamp"].isna().sum())
print("Invalid timestamps:", invalid_timestamp_rows)

work = work.loc[work["_parsed_timestamp"].notna()].copy()
work = work.sort_values("_parsed_timestamp", kind="mergesort").reset_index(drop=True)

n = len(work)
train_end = int(n * 0.70)
valid_end = int(n * 0.80)

train_df = work.iloc[:train_end].copy()
valid_df = work.iloc[train_end:valid_end].copy()
test_df = work.iloc[valid_end:].copy()

for split_name, split_df in [
    ("Train", train_df), ("Validation", valid_df), ("Test", test_df)
]:
    print(
        f"{split_name}: n={len(split_df):,}; "
        f"{split_df['_parsed_timestamp'].min()} -> {split_df['_parsed_timestamp'].max()}"
    )
    display(
        split_df[TARGET].value_counts(normalize=True)
        .mul(100).round(2).rename("percent").to_frame()
    )

classes = sorted(work[TARGET].astype(str).unique())
print("Classes:", classes)
if set(train_df[TARGET].astype(str).unique()) != set(classes):
    raise ValueError("At least one class is absent from training data; revise split before modeling.")
if set(valid_df[TARGET].astype(str).unique()) != set(classes):
    warnings.warn("Validation partition does not contain every class.")
if set(test_df[TARGET].astype(str).unique()) != set(classes):
    warnings.warn("Test partition does not contain every class.")

## 4. Preprocessing and class imbalance strategy

All preprocessing is inside a scikit-learn `Pipeline`, so imputers, scalers, and encoders are fitted on the training partition only.

The initial comparison uses:
- Logistic Regression with balanced class weights.
- Random Forest with balanced subsample weights.
- LightGBM and XGBoost only if already installed in the Colab runtime.

No SMOTE or resampling is applied in this notebook. This avoids synthetic samples crossing chronological boundaries. If resampling is later tested, it must occur only inside the training partition.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, classification_report,
    confusion_matrix, log_loss, average_precision_score,
    roc_auc_score, precision_recall_curve
)

def make_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def make_preprocessor(X):
    numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot())
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("categorical", categorical_pipe, categorical_cols)
    ])

def make_model_pipeline(X_train, model_name):
    prep = make_preprocessor(X_train)
    if model_name == "Dummy":
        estimator = DummyClassifier(strategy="prior")
    elif model_name == "LogisticRegression":
        estimator = LogisticRegression(
            max_iter=2500, class_weight="balanced",
            solver="lbfgs", random_state=42
        )
    elif model_name == "RandomForest":
        estimator = RandomForestClassifier(
            n_estimators=350, min_samples_leaf=4,
            class_weight="balanced_subsample",
            random_state=42, n_jobs=-1
        )
    else:
        raise ValueError(model_name)
    return Pipeline([("preprocess", prep), ("model", estimator)])

## 5. Fit models and compare on validation data

In [ ]:
X_train = train_df[FEATURES].copy()
X_valid = valid_df[FEATURES].copy()
X_test = test_df[FEATURES].copy()

y_train = train_df[TARGET].astype(str)
y_valid = valid_df[TARGET].astype(str)
y_test = test_df[TARGET].astype(str)

MODEL_NAMES = ["Dummy", "LogisticRegression", "RandomForest"]
fitted_models = {}
validation_rows = []

for model_name in MODEL_NAMES:
    print("Fitting:", model_name)
    pipe = make_model_pipeline(X_train, model_name)
    pipe.fit(X_train, y_train)
    fitted_models[model_name] = pipe

    pred = pipe.predict(X_valid)
    row = {
        "model": model_name,
        "accuracy": accuracy_score(y_valid, pred),
        "balanced_accuracy": balanced_accuracy_score(y_valid, pred),
        "macro_precision": precision_score(y_valid, pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_valid, pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_valid, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_valid, pred, average="weighted", zero_division=0),
    }
    validation_rows.append(row)

validation_results = pd.DataFrame(validation_rows).sort_values(
    ["macro_f1", "balanced_accuracy"], ascending=False
)
display(validation_results)

## 6. Select a model using validation only, then evaluate the untouched test set

The selected model is chosen by validation macro-F1, with balanced accuracy as a tie-breaker. This selection rule is fixed before looking at test performance.

In [ ]:
best_model_name = validation_results.iloc[0]["model"]
best_model = fitted_models[best_model_name]
print("Selected using validation only:", best_model_name)

test_pred = best_model.predict(X_test)
test_probs = best_model.predict_proba(X_test) if hasattr(best_model, "predict_proba") else None

test_summary = {
    "selected_model": best_model_name,
    "accuracy": accuracy_score(y_test, test_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
    "macro_precision": precision_score(y_test, test_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(y_test, test_pred, average="macro", zero_division=0),
    "macro_f1": f1_score(y_test, test_pred, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_test, test_pred, average="weighted", zero_division=0),
}
display(pd.Series(test_summary).to_frame("test_result"))

label_order = sorted(set(y_train) | set(y_valid) | set(y_test))
cm = confusion_matrix(y_test, test_pred, labels=label_order)
cm_df = pd.DataFrame(cm, index=label_order, columns=label_order)
print("Test confusion matrix:")
display(cm_df)

print("Test classification report:")
test_report = pd.DataFrame(
    classification_report(y_test, test_pred, labels=label_order,
                          output_dict=True, zero_division=0)
).T
display(test_report)

## 7. High-risk threshold selection on validation data

This section treats `high_risk` as the positive class in a one-versus-rest screening task.

A threshold is selected using **validation probabilities only**. The default policy is to choose the threshold that maximizes validation F1 subject to validation recall of at least 80%, if such a threshold exists. If no threshold meets that recall target, choose the threshold with the highest validation F1 and explicitly report that the recall constraint was not achieved.

The chosen threshold is then evaluated once on the test partition.

In [ ]:
if "high_risk" not in best_model.classes_:
    raise ValueError("Selected model does not contain the high_risk class.")

high_idx = list(best_model.classes_).index("high_risk")
valid_high_prob = best_model.predict_proba(X_valid)[:, high_idx]
test_high_prob = best_model.predict_proba(X_test)[:, high_idx]

y_valid_high = (y_valid == "high_risk").astype(int)
y_test_high = (y_test == "high_risk").astype(int)

prec, rec, thresholds = precision_recall_curve(y_valid_high, valid_high_prob)
# precision_recall_curve has one more precision/recall point than thresholds.
pr_rows = []
for i, threshold in enumerate(thresholds):
    p, r = prec[i], rec[i]
    f1v = 0.0 if (p + r) == 0 else 2 * p * r / (p + r)
    pr_rows.append({"threshold": float(threshold), "precision": float(p),
                    "recall": float(r), "f1": float(f1v)})

threshold_table = pd.DataFrame(pr_rows)
eligible = threshold_table[threshold_table["recall"] >= 0.80]
recall_constraint_met = not eligible.empty
if recall_constraint_met:
    selected_threshold = float(eligible.sort_values(
        ["f1", "precision"], ascending=False
    ).iloc[0]["threshold"])
else:
    selected_threshold = float(threshold_table.sort_values(
        ["f1", "recall"], ascending=False
    ).iloc[0]["threshold"])

print("Selected threshold from validation:", selected_threshold)
print("Validation recall >= 0.80 constraint met:", recall_constraint_met)

test_high_pred = (test_high_prob >= selected_threshold).astype(int)
high_risk_test_metrics = {
    "threshold_selected_on_validation": selected_threshold,
    "validation_recall_constraint_met": recall_constraint_met,
    "test_precision": precision_score(y_test_high, test_high_pred, zero_division=0),
    "test_recall": recall_score(y_test_high, test_high_pred, zero_division=0),
    "test_f1": f1_score(y_test_high, test_high_pred, zero_division=0),
    "test_average_precision": average_precision_score(y_test_high, test_high_prob),
    "test_roc_auc": roc_auc_score(y_test_high, test_high_prob),
}
display(pd.Series(high_risk_test_metrics).to_frame("result"))

display(pd.DataFrame(
    confusion_matrix(y_test_high, test_high_pred, labels=[0, 1]),
    index=["actual_not_high_risk", "actual_high_risk"],
    columns=["predicted_not_high_risk", "predicted_high_risk"]
))

## 8. Optional LightGBM and XGBoost comparison

These libraries are optional. If installed, this section fits one fixed configuration of each model and evaluates it on validation data. If not installed, the notebook records that fact and continues.

For reproducibility, do not install arbitrary packages or tune parameters based on the test set. The same training-only preprocessing is retained.

In [ ]:
optional_validation_rows = []
optional_models = {}

# LightGBM
if importlib.util.find_spec("lightgbm") is not None:
    from lightgbm import LGBMClassifier
    lgb_prep = make_preprocessor(X_train)
    lgb_pipe = Pipeline([
        ("preprocess", lgb_prep),
        ("model", LGBMClassifier(
            n_estimators=350, learning_rate=0.04, num_leaves=31,
            class_weight="balanced", random_state=42, verbosity=-1
        ))
    ])
    lgb_pipe.fit(X_train, y_train)
    lgb_pred = lgb_pipe.predict(X_valid)
    optional_models["LightGBM"] = lgb_pipe
    optional_validation_rows.append({
        "model": "LightGBM",
        "accuracy": accuracy_score(y_valid, lgb_pred),
        "balanced_accuracy": balanced_accuracy_score(y_valid, lgb_pred),
        "macro_f1": f1_score(y_valid, lgb_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_valid, lgb_pred, average="weighted", zero_division=0)
    })
else:
    print("LightGBM not installed; skipped.")

# XGBoost
if importlib.util.find_spec("xgboost") is not None:
    from xgboost import XGBClassifier
    xgb_prep = make_preprocessor(X_train)
    xgb_pipe = Pipeline([
        ("preprocess", xgb_prep),
        ("model", XGBClassifier(
            n_estimators=350, max_depth=5, learning_rate=0.04,
            subsample=0.85, colsample_bytree=0.85,
            objective="multi:softprob", eval_metric="mlogloss",
            random_state=42, n_jobs=-1
        ))
    ])
    xgb_pipe.fit(X_train, y_train)
    xgb_pred = xgb_pipe.predict(X_valid)
    optional_models["XGBoost"] = xgb_pipe
    optional_validation_rows.append({
        "model": "XGBoost",
        "accuracy": accuracy_score(y_valid, xgb_pred),
        "balanced_accuracy": balanced_accuracy_score(y_valid, xgb_pred),
        "macro_f1": f1_score(y_valid, xgb_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_valid, xgb_pred, average="weighted", zero_division=0)
    })
else:
    print("XGBoost not installed; skipped.")

optional_validation_results = pd.DataFrame(optional_validation_rows)
display(optional_validation_results if not optional_validation_results.empty else
        pd.DataFrame({"note": ["No optional gradient-boosting libraries were available."]}))

## 9. Robustness check: repeated chronological expanding-window evaluation

This section checks whether the primary Random Forest and Logistic Regression results are unusually dependent on one particular chronological cutoff. It uses expanding-window folds: each later validation block follows its training block in time.

Each fold fits preprocessing and the model from scratch on that fold's training data. No future records are used to fit earlier folds.

In [ ]:
def expanding_window_evaluation(model_name, n_splits=3, initial_train_fraction=0.50):
    ordered = work.reset_index(drop=True)
    n_all = len(ordered)
    initial_end = int(n_all * initial_train_fraction)
    remaining = n_all - initial_end
    block = remaining // n_splits
    rows = []

    for fold in range(n_splits):
        tr_end = initial_end + fold * block
        va_end = initial_end + (fold + 1) * block if fold < n_splits - 1 else n_all
        fold_train = ordered.iloc[:tr_end]
        fold_valid = ordered.iloc[tr_end:va_end]

        if fold_train[TARGET].nunique() < 2 or fold_valid[TARGET].nunique() < 2:
            print(f"Skipping fold {fold+1}: insufficient class diversity.")
            continue

        Xtr = fold_train[FEATURES].copy()
        Xva = fold_valid[FEATURES].copy()
        ytr = fold_train[TARGET].astype(str)
        yva = fold_valid[TARGET].astype(str)

        fold_pipe = make_model_pipeline(Xtr, model_name)
        fold_pipe.fit(Xtr, ytr)
        pred = fold_pipe.predict(Xva)

        rows.append({
            "model": model_name,
            "fold": fold + 1,
            "train_rows": len(fold_train),
            "validation_rows": len(fold_valid),
            "train_end": fold_train["_parsed_timestamp"].max(),
            "validation_start": fold_valid["_parsed_timestamp"].min(),
            "balanced_accuracy": balanced_accuracy_score(yva, pred),
            "macro_f1": f1_score(yva, pred, average="macro", zero_division=0),
            "high_risk_recall": recall_score(
                yva, pred, labels=["high_risk"], average="macro", zero_division=0
            ) if "high_risk" in set(yva) else np.nan
        })
    return rows

robustness_rows = []
for model_name in ["LogisticRegression", "RandomForest"]:
    robustness_rows.extend(expanding_window_evaluation(model_name))

robustness_results = pd.DataFrame(robustness_rows)
display(robustness_results)
if not robustness_results.empty:
    display(robustness_results.groupby("model")[
        ["balanced_accuracy", "macro_f1", "high_risk_recall"]
    ].agg(["mean", "std"]).round(4))

## 10. Validation-only permutation importance

Permutation importance is calculated on the validation partition, not the final test set. It estimates how much validation macro-F1 falls when one feature is shuffled. Correlated features can share importance, and permutation importance does not prove causality or feature provenance.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model, X_valid, y_valid,
    scoring="f1_macro", n_repeats=8,
    random_state=42, n_jobs=-1
)
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance_df)

## 11. Export all outputs

In [ ]:
from pathlib import Path
Path("outputs_07").mkdir(exist_ok=True)

validation_results.to_csv("outputs_07/07_validation_model_comparison.csv", index=False)
pd.Series(test_summary).to_frame("test_result").to_csv("outputs_07/07_final_test_summary.csv")
cm_df.to_csv("outputs_07/07_final_test_confusion_matrix.csv")
test_report.to_csv("outputs_07/07_final_test_classification_report.csv")
threshold_table.to_csv("outputs_07/07_validation_high_risk_threshold_curve.csv", index=False)
pd.Series(high_risk_test_metrics).to_frame("result").to_csv("outputs_07/07_high_risk_test_threshold_metrics.csv")
optional_validation_results.to_csv("outputs_07/07_optional_boosting_validation_results.csv", index=False)
robustness_results.to_csv("outputs_07/07_expanding_window_robustness.csv", index=False)
importance_df.to_csv("outputs_07/07_validation_permutation_importance.csv", index=False)

metadata = {
    "source_file": DATA_PATH.name,
    "source_sha256": source_sha256,
    "rows_loaded": int(len(df)),
    "columns_loaded": int(df.shape[1]),
    "invalid_timestamps_excluded": invalid_timestamp_rows,
    "target": TARGET,
    "features": FEATURES,
    "excluded_fields": EXCLUDED_FIELDS,
    "split": "chronological 70% train / 10% validation / 20% test",
    "model_selection": "validation macro-F1, balanced accuracy tie-breaker",
    "high_risk_threshold_selection": "validation F1 maximization subject to recall >= 0.80 where feasible",
    "selected_model": best_model_name,
    "selected_high_risk_threshold": selected_threshold,
    "models_compared": MODEL_NAMES + list(optional_models.keys()),
    "limitations": [
        "Predicts Kaggle-provided risk labels, not independently verified fraud outcomes.",
        "Remaining features' independence from label-generation rules is not established by this notebook.",
        "Test set is a single chronological holdout."
    ]
}
Path("outputs_07/07_reproducibility_metadata.json").write_text(json.dumps(metadata, indent=2, default=str))
print("Saved outputs to outputs_07/")

## 12. Interpretation checklist and reporting gate

Before making a research claim, answer:

- [ ] Did the model outperform the Dummy baseline on validation and final test?
- [ ] What are macro-F1 and balanced accuracy on the untouched test set?
- [ ] What is high-risk recall at the threshold selected using validation only?
- [ ] What precision / false-positive burden accompanies that recall?
- [ ] Are expanding-window results reasonably stable?
- [ ] Which features matter on validation, and are their origins documented?
- [ ] Were optional LightGBM/XGBoost models available, or skipped?
- [ ] Is the test period later than every training and validation record?
- [ ] Are all claims framed as prediction of Kaggle risk labels rather than verified real-world fraud?

### Research conclusion boundary

A strong score does not prove absence of leakage. A lower score using defensible features may be more credible than perfect classification using a field that encodes the target. Do not call this a production-ready ECIPS detector without independent labels, a documented prediction-time feature set, and evaluation on an external dataset.